# IBVAP ML Workflow

This notebook is an execution guide for real CCTV data. It does not create datasets, fabricate metrics, or claim training until the training cell is run with a real dataset.

## 1. Project / ML objective
Use pretrained YOLO for person and vehicle detection, ByteTrack/BoT-SORT for tracking, pretrained face/embedding models when configured, PaddleOCR for probabilistic ANPR, and geometry/rule modules for intrusion, loitering, night movement, and baseline activity events.

## 2. Environment and dependency checks
## 3. GPU/CUDA availability check
## 4. Imports
## 5. Configuration
## 6. Dataset paths
## 7. Dataset inspection
## 8. Dataset visualization
## 9. Dataset preparation
## 10. YOLO model loading
## 11. YOLO training / fine-tuning
## 12. YOLO validation
## 13. Metrics
## 14. Test predictions
## 15. Video inference
## 16. Tracking
## 17. Virtual-fence testing
## 18. Loitering testing
## 19. Night movement testing
## 20. ANPR testing
## 21. Face detection / recognition testing
## 22. Activity / anomaly module testing
## 23. Full pipeline testing
## 24. JSON event generation
## 25. Results and conclusions

In [ ]:
# 2-5. Environment, GPU, imports, and configuration
from pathlib import Path
import json
import importlib.util
import sys

import yaml

ML_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ML_ROOT) not in sys.path:
    sys.path.insert(0, str(ML_ROOT))

from src.activity import ActivityMonitor
from src.anpr import ANPRPipeline
from src.detector import YOLODetector, select_device
from src.face_recognition import FaceRecognizer
from src.intrusion import VirtualFence
from src.tracker import ObjectTracker

CONFIG_PATH = ML_ROOT / "config.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
DATASET_YAML = "path/to/dataset/data.yaml"
MODEL_NAME = CONFIG["model_path"]
EPOCHS = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
CONFIDENCE_THRESHOLD = CONFIG["confidence_threshold"]
DEVICE = select_device(CONFIG.get("device"))
print({"python": sys.version.split()[0], "device": DEVICE, "yaml_exists": CONFIG_PATH.exists()})
print("ultralytics installed:", importlib.util.find_spec("ultralytics") is not None)

In [ ]:
# 6-9. Dataset paths, inspection, visualization, and preparation
from pathlib import Path

DATASET_ROOT = Path(DATASET_YAML).expanduser().resolve().parent
if not Path(DATASET_YAML).expanduser().exists():
    print("Configure DATASET_YAML before dataset inspection; no dataset was fabricated.")
else:
    dataset_config = yaml.safe_load(Path(DATASET_YAML).expanduser().read_text(encoding="utf-8"))
    print("classes:", dataset_config.get("names"))
    for split in ("train", "val", "test"):
        image_dir = DATASET_ROOT / "images" / split
        label_dir = DATASET_ROOT / "labels" / split
        print(split, {"images": len(list(image_dir.glob("*"))) if image_dir.exists() else 0,
                      "labels": len(list(label_dir.glob("*.txt"))) if label_dir.exists() else 0})

# Visualization is intentionally data-dependent. Set SAMPLE_IMAGE to a real file to inspect it.
SAMPLE_IMAGE = None
if SAMPLE_IMAGE:
    from IPython.display import display
    from PIL import Image
    display(Image.open(SAMPLE_IMAGE))

In [ ]:
# 10-13. YOLO loading, manual training, validation, and metrics
from ultralytics import YOLO

model = YOLO(MODEL_NAME)
print("Loaded pretrained model:", MODEL_NAME)

# Training is deliberately an explicit manual action. Run only after setting a real DATASET_YAML.
def train_model():
    if not Path(DATASET_YAML).expanduser().exists():
        raise FileNotFoundError(f"Set DATASET_YAML to a real file: {DATASET_YAML}")
    return model.train(data=DATASET_YAML, epochs=EPOCHS, imgsz=IMAGE_SIZE,
                       batch=BATCH_SIZE, conf=CONFIDENCE_THRESHOLD, device=DEVICE,
                       project=str(ML_ROOT / "runs"), name="ibvap_yolo")

# Uncomment to start training manually; this cell does not train by itself.
# training_results = train_model()

# After training, set a real best.pt path before running validation.
BEST_WEIGHTS = ML_ROOT / "models" / "best.pt"
if BEST_WEIGHTS.exists():
    trained_model = YOLO(str(BEST_WEIGHTS))
    validation_results = trained_model.val(data=DATASET_YAML, imgsz=IMAGE_SIZE, device=DEVICE)
    print("Validation executed; inspect validation_results for actual metrics.")
else:
    print("No best.pt found; validation and metrics are not claimed.")

In [ ]:
# 14-16. Test predictions, video inference, and tracking
IMAGE_PATH = None
VIDEO_PATH = None
inference_model = trained_model if "trained_model" in globals() else model

if IMAGE_PATH:
    predictions = inference_model.predict(IMAGE_PATH, conf=CONFIDENCE_THRESHOLD, device=DEVICE)
    print("Predictions generated for configured image:", IMAGE_PATH)
else:
    print("Set IMAGE_PATH to run test predictions.")

if VIDEO_PATH:
    video_predictions = inference_model.predict(source=VIDEO_PATH, stream=True,
                                                conf=CONFIDENCE_THRESHOLD, device=DEVICE)
    for frame_index, result in enumerate(video_predictions):
        print("video frame", frame_index, "detections", len(result.boxes))
        if frame_index >= 4:
            break
else:
    print("Set VIDEO_PATH to run video inference.")

detector = YOLODetector(MODEL_NAME, CONFIDENCE_THRESHOLD, DEVICE, CONFIG.get("classes"))
tracker = ObjectTracker(detector)
if IMAGE_PATH:
    tracks = tracker.track(IMAGE_PATH)
    print("tracks:", tracks)

In [ ]:
# 17-19. Virtual fence, loitering, and night movement
fence = VirtualFence(CONFIG.get("restricted_zones", {}))
activity = ActivityMonitor(CONFIG["loitering_seconds"], tuple(CONFIG["night_hours"]))

sample_tracks = []
if sample_tracks:
    print("intrusion events:", fence.evaluate(sample_tracks))
    print("activity events:", activity.evaluate(sample_tracks))
else:
    print("Provide real tracker output to test fence and activity rules.")

# These are configurable rules, not a trained suspicious-activity model.

In [ ]:
# 20-22. ANPR, face, and activity/anomaly module testing
# Configure these adapters with real models before testing. No OCR text or identity is fabricated.
anpr = ANPRPipeline()
face = FaceRecognizer()
print("ANPR without configured detector/OCR:", anpr.read(None))
print("Face module without configured detector:", face.analyze(None))
print("Rule-based activity monitor is ready; future anomaly models can implement the same event interface.")

In [ ]:
# 23-25. Full pipeline, JSON event generation, and conclusions
from datetime import datetime, timezone

camera_id = "CAM_001"
all_events = []
if IMAGE_PATH:
    current_tracks = tracker.track(IMAGE_PATH)
    all_events = fence.evaluate(current_tracks) + activity.evaluate(current_tracks)

now = datetime.now(timezone.utc).isoformat()
for event in all_events:
    event.update({"camera_id": camera_id, "timestamp": now})

print(json.dumps({"camera_id": camera_id, "timestamp": now,
                  "events": all_events}, indent=2))
print("Conclusion: configure real data and weights, run training manually if needed, then evaluate actual outputs and metrics.")